In [1]:
!pip install pandas numpy matplotlib seaborn

In [2]:
from google.colab import files

uploaded = files.upload()

Saving crop_production.csv to crop_production (1).csv
Saving Market_prices.csv to Market_prices (1).csv


In [3]:
import pandas as pd

crop_df = pd.read_csv("crop_production.csv")
market_df = pd.read_csv("Market_prices.csv")

print(crop_df.shape)
print(market_df.shape)

crop_df.head()

(19689, 10)
(10819, 11)


,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield
0,Arecanut,1997,Whole Year,Assam,73814.0,56708,2051.4,7024878.38,22882.34,0.796087
1,Arhar/Tur,1997,Kharif,Assam,6637.0,4685,2051.4,631643.29,2057.47,0.710435
2,Castor seed,1997,Kharif,Assam,796.0,22,2051.4,75755.32,246.76,0.238333
3,Coconut,1997,Whole Year,Assam,19656.0,126905000,2051.4,1870661.52,6093.36,5238.051739
4,Cotton(lint),1997,Kharif,Assam,1739.0,794,2051.4,165500.63,539.09,0.420909


In [4]:
crop_df.info()

crop_df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19689 entries, 0 to 19688
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Crop             19689 non-null  object 
 1   Crop_Year        19689 non-null  int64  
 2   Season           19689 non-null  object 
 3   State            19689 non-null  object 
 4   Area             19689 non-null  float64
 5   Production       19689 non-null  int64  
 6   Annual_Rainfall  19689 non-null  float64
 7   Fertilizer       19689 non-null  float64
 8   Pesticide        19689 non-null  float64
 9   Yield            19689 non-null  float64
dtypes: float64(5), int64(2), object(3)
memory usage: 1.5+ MB


,0
Crop,0
Crop_Year,0
Season,0
State,0
Area,0
Production,0
Annual_Rainfall,0
Fertilizer,0
Pesticide,0
Yield,0


In [5]:
crop_df.columns

Index(['Crop', 'Crop_Year', 'Season', 'State', 'Area', 'Production',
       'Annual_Rainfall', 'Fertilizer', 'Pesticide', 'Yield'],
      dtype='object')

In [6]:
market_df.info()

market_df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10819 entries, 0 to 10818
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   State           10819 non-null  object 
 1   District        10819 non-null  object 
 2   Market          10819 non-null  object 
 3   Commodity       10819 non-null  object 
 4   Variety         10819 non-null  object 
 5   Grade           10819 non-null  object 
 6   Arrival_Date    10819 non-null  object 
 7   Min_Price       10819 non-null  int64  
 8   Max_Price       10819 non-null  int64  
 9   Modal_Price     10819 non-null  float64
 10  Commodity_Code  10819 non-null  int64  
dtypes: float64(1), int64(3), object(7)
memory usage: 929.9+ KB


,0
State,0
District,0
Market,0
Commodity,0
Variety,0
Grade,0
Arrival_Date,0
Min_Price,0
Max_Price,0
Modal_Price,0


In [7]:
crop_names = set(crop_df["Crop"].str.lower().unique())

commodity_names = set(
    market_df["Commodity"].astype(str).str.lower().unique()
)

common = crop_names.intersection(commodity_names)

print("Common Crops:", len(common))
print(sorted(list(common))[:50])

Common Crops: 5
['banana', 'maize', 'onion', 'potato', 'wheat']


In [8]:
market_df["Arrival_Date"] = pd.to_datetime(
    market_df["Arrival_Date"],
    errors="coerce"
)

In [9]:
price_df = (
    market_df
    .groupby("Commodity")["Modal_Price"]
    .mean()
    .reset_index()
)

price_df.head()

,Commodity,Modal_Price
0,Apple,8972.039952
1,Bajra(Pearl Millet/Cumbu),2049.890110
2,Banana,2452.813853
3,Barley (Jau),2028.076923
4,Barley(Jau),2041.250000


In [10]:
crop_mapping = {
    "Cotton(lint)": "Cotton",
    "Arhar/Tur": "Tur"
}

In [11]:
crop_df["Crop"] = crop_df["Crop"].replace(crop_mapping)

In [13]:
merged_df = pd.merge(
    crop_df,
    price_df,
    left_on="Crop",
    right_on="Commodity",
    how="inner"
)

In [14]:
merged_df.head()

,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield,Commodity,Modal_Price
0,Cotton,1997,Kharif,Assam,1739.0,794,2051.4,165500.63,539.09,0.420909,Cotton,7018.952055
1,Maize,1997,Kharif,Assam,19216.0,14721,2051.4,1828786.72,5956.96,0.615652,Maize,2600.000000
2,Onion,1997,Whole Year,Assam,7832.0,17943,2051.4,745371.44,2427.92,2.342609,Onion,1464.607920
3,Potato,1997,Whole Year,Assam,75259.0,671871,2051.4,7162399.03,23330.29,7.561304,Potato,931.947187
4,Wheat,1997,Rabi,Assam,84698.0,110054,2051.4,8060708.66,26256.38,1.259524,Wheat,2462.426606


In [15]:
merged_df["Revenue"] = (
    merged_df["Production"]
    * merged_df["Modal_Price"]
)

In [16]:
merged_df["Cost"] = (
    merged_df["Fertilizer"]
    + merged_df["Pesticide"]
)

In [17]:
merged_df["Profit"] = (
    merged_df["Revenue"]
    - merged_df["Cost"]
)

In [18]:
merged_df.to_csv(
    "processed_data.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully
